# News Agent — Agente de Noticias Financieras

Agente que recolecta, filtra y resume noticias financieras desde feeds RSS de medios internacionales y regionales.

| Versión | Descripción | Estado |
|---------|-------------|--------|
| 0 | Lector de feeds RSS: título, enlace y fecha | COMPLETADA |
| 1 | Filtrado por palabras clave y exportación a CSV | COMPLETADA |
| 2 | Almacenamiento en base de datos y deduplicación | Pendiente |
| 3 | Análisis de sentimiento con NLP | Pendiente |
| 4 | Agente con LLM: resumen y alertas automáticas | Pendiente |

In [1]:
print("Hello World")
name = input("Dime tu nombre")
print(name)

Hello World
Santiago


In [2]:
# Versión 0 — News Agent: lector de feeds RSS de medios financieros.
# Crea la estructura de carpetas del proyecto si aún no existe.
# - data/raw/       → feeds RSS descargados sin procesar
# - data/processed/ → noticias limpias y filtradas
# - logs/           → registros de ejecución
# - config/         → fuentes RSS y parámetros de configuración

from pathlib import Path
import os

base = Path(os.getcwd())
print(f"Creando carpetas en: {base}")

for carpeta in ["data/raw", "data/processed", "logs", "config"]:
    ruta = base / carpeta
    ruta.mkdir(parents=True, exist_ok=True)
    print(f"✓ {ruta}")

#Aquí se crearon las carpetas donde el agente se pondrá a almacenar la información de noticias que vaya recolectando

Creando carpetas en: g:\Data Lake\Noticias
✓ g:\Data Lake\Noticias\data\raw
✓ g:\Data Lake\Noticias\data\processed
✓ g:\Data Lake\Noticias\logs
✓ g:\Data Lake\Noticias\config


In [3]:
#Ahora instalaremos Feedparser
import feedparser
print(f"feedparser {feedparser.__version__} importado correctamente.")

feedparser 6.0.12 importado correctamente.


In [4]:
# Fuentes RSS del agente — medios financieros por cobertura geográfica

FEEDS = [
    {"nombre": "Reuters Business",  "url": "https://news.google.com/rss/search?q=when:24h+allinurl:reuters.com&ceid=US:en&hl=en-US&gl=US"},    # Internacional
    {"nombre": "Yahoo Finance",     "url": "https://finance.yahoo.com/news/rssindex"},            # Internacional (EE.UU.)
    {"nombre": "Expansión México",  "url": "https://expansion.mx/rss"},                           # México
    {"nombre": "Investing España",        "url": "https://es.investing.com/rss/news.rss"}, # España
]

In [ ]:
# Recorre cada fuente RSS y muestra las primeras 5 noticias

for feed in FEEDS:

    # Descarga y parsea el feed
    resultado = feedparser.parse(feed["url"])

    # Encabezado del medio
    print(f"\n{'='*60}")
    print(f"  {feed['nombre']}")
    print(f"{'='*60}")

    # Itera sobre las primeras 5 entradas
    for entrada in resultado.entries[:5]:
        titulo = entrada.get("title", "Sin título")
        enlace = entrada.get("link", "Sin enlace")
        fecha  = entrada.get("published", "Fecha no disponible")

        print(f"\nTítulo : {titulo}")
        print(f"Enlace : {enlace}")
        print(f"Fecha  : {fecha}")
        print("-" * 60)

In [6]:
import subprocess, sys

# Instala python-dotenv si no está disponible en el entorno
subprocess.check_call([sys.executable, "-m", "pip", "install", "python-dotenv", "-q"])

import os
from dotenv import load_dotenv

# Carga las variables definidas en el archivo .env al entorno del proceso
load_dotenv()

# Lee la API key de Anthropic desde las variables de entorno
ANTHROPIC_API_KEY = os.getenv("ANTHROPIC_API_KEY")

# Confirma que la clave se cargó, mostrando solo los primeros 8 caracteres
if ANTHROPIC_API_KEY:
    print(f"ANTHROPIC_API_KEY cargada correctamente: {ANTHROPIC_API_KEY[:8]}...")
else:
    print("⚠️  ANTHROPIC_API_KEY no encontrada. Verifica que exista en el archivo .env")

ANTHROPIC_API_KEY cargada correctamente: sk-ant-a...


In [ ]:
import anthropic
client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)

# --- Recolección de titulares ---
# Recorre todos los feeds y acumula hasta 20 títulos en total
titulos = []
for feed in FEEDS:
    resultado = feedparser.parse(feed["url"])
    contador = 0
    for entrada in resultado.entries:
        if contador >= 5:
            break
        titulo = entrada.get("title", "").strip()
        if titulo:
            titulos.append(f"[{feed['nombre']}] {titulo} | URL: {entrada.get('link', 'No disponible')}")
            contador += 1

# Esta noticias_texto será la lista previamente creada de encabezados de noticias
# Para esta ocasión, lo limitamos a 5 noticias de las 4 fuentes, dándo un total de 20 noticas
noticias_texto = "\n".join(f"{i+1}. {t}" for i, t in enumerate(titulos))

#Todo para que el prompt le pida al agente que de esas 20 noticias
# se seleccionen las 8 más relevantes para los temas que menciona el prompr
#También le pedimos que incluya el url de la notica por si el usuario desea acceder y leerela completa
prompt = f"""You are a senior financial analyst. I am providing you with a list of headlines from various sources (Reuters Business, Yahoo Finance, Expansión México, and Investing España) riight bellow: You must analyze the entire list provided, regardless of the source, and do not filter by source.

{noticias_texto}

Your task:
1. Select the 8 most relevant news for a finance professional interested in global markets, political situations, the economy, and stock markets.
2. For each selected news item, write a summary of exactly 2 sentences: the first explains the fact, the second explains its financial implication.
3. If the headline is in English, write the summary in English. If it is in Spanish, write it in Spanish.
4. Include the exact URL of the news item you are summarizing.
5. Respond ONLY with the 8 numbered news items, without any introduction or additional text.

Response format:
1. [Headline]
   Summary: [Sentence 1. Sentence 2.]
   URL: [Link]
"""

# --- Llamada a Anthropic y presentación del resultado ---
print("Analizando titulares con Claude...\n")
print("=" * 60)
respuesta = client.messages.create(
    model="claude-haiku-4-5-20251001",
    #estamos usando de momento el modelo más económico en tokens para las pruebas
    max_tokens=2048,
    #De momento ponemos un máximo en el  número de tokens porque solo se está enfocando en hacer un resumen de las noticias como si fuera encabezado
    messages=[{"role": "user", "content": prompt}]
)
print(respuesta.content[0].text)
print("=" * 60)

Analizando titulares con Claude...

1. [Yahoo Finance] 47-year-old high-end steak and seafood chain closes 80 locations
   Summary: A long-established upscale restaurant chain is shuttering 80 locations, indicating significant operational challenges within the fine dining sector. This reflects broader consumer spending pressures on discretionary dining and potential profitability issues in the hospitality industry.
   URL: https://finance.yahoo.com/economy/articles/47-old-high-end-steak-170700893.html

2. [Expansión México] Infraestructura eléctrica pone a prueba auge de los centros de datos en México
   Summary: Mexico's electrical infrastructure is becoming a bottleneck for the rapid expansion of data centers in the country. Este desafío representa un riesgo operativo y de inversión para las empresas tecnológicas que buscan expandirse en México y podría afectar la competitividad del país en la economía digital global.
   URL: https://expansion.mx/empresas/2026/05/29/infraestructura-e

In [18]:
print(noticias_texto)

1. [Reuters Business] Dell shares soar 30% as AI server demand, price hikes power stellar quarter - Reuters | URL: https://news.google.com/rss/articles/CBMivAFBVV95cUxPcVJqdDdzRnFib0ZkeU1rd1F6WlZrOHF4cTB2MGVoLUhqRlNJNzY1RXY0MHBpcHhNUXBQODVMSlNRd0c5N292QnRTcUZpRkpvR2dVYnVLSm1YalVEUkZ0blV6UllTZTJPaUtlU2VfbDZjWk1ESWYwcGtwRzAzWHV3UjN0dS1Oc1ExNXNLSHVmRmRBQy1xOWFyN3ZaNjFFdDVzYWxEcGN4UTBmdUs1bV9ySnpBaTVRSkdQMkZqcg?oc=5
2. [Reuters Business] Wall Street follows global stocks higher, crude prices ease on potential US-Iran truce extension - Reuters | URL: https://news.google.com/rss/articles/CBMiiAFBVV95cUxOWTVlS2Y4YzBmWGowUkNTSF94M21QQjZ1TzViNVJhaEgwZFBJa200R1hydjhQYWgzUXJlcENFSjlaZndHVms5eXd5VW5XT3ljUllpcVZ1emhFT1plNUhLVzNJOEg2Wi1zUVRkUzZCVENaVS1sQWhTN2V4cVZPXzZmY1VpcmpBeTFh?oc=5
3. [Reuters Business] Rule changes may only yield modest Fed balance sheet contraction - Reuters | URL: https://news.google.com/rss/articles/CBMiuAFBVV95cUxNOGpTNFducWRsSzd4eHZEeDItYnJ6bEVnczhZOEJtbENsTWdSS1RhMUNKUmJl